# 05 — AQI Feature Engineering and Linear Regression (Google Colab)


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

In [4]:
# Project paths
PROJECT_ROOT = Path("/content/drive/MyDrive/Internship_AQI_Prediction")
PATH_MODEL_DATA = ( PROJECT_ROOT / "data" / "processed" / "df_model.csv" )
PATH_FEATURED_DATA = ( PROJECT_ROOT / "data" / "processed" / "df_model_featured.csv" )

## 1. Load only the columns needed for the baseline

The full CSV contains many columns we do not need for this first model. Loading only the required columns reduces memory use. This does not modify or delete the original CSV.

In [6]:
# Read the header first to select only columns that exist.
available_columns = pd.read_csv(PATH_MODEL_DATA, nrows=0).columns.tolist()

required_columns = [
    "StationId", "Datetime", "AQI_final",
    "Target_AQI_24h", "Target_AQI_48h"
]

missing_required = [
    col for col in required_columns if col not in available_columns
]

if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

# Candidate measurements: use only those present in the CSV.
candidate_features = [
    "PM2.5", "PM10", "NO2", "SO2", "O3", "CO",
    "Temperature_C", "Relative_Humidity_pct", "Wind_Speed_kmh"
]

load_columns = list(dict.fromkeys(
    required_columns +
    [col for col in candidate_features if col in available_columns]
))

df_model = pd.read_csv(
    PATH_MODEL_DATA,
    usecols=load_columns,
    low_memory=False
)

df_model["Datetime"] = pd.to_datetime(
    df_model["Datetime"], errors="coerce"
)

df_model = df_model.dropna(
    subset=required_columns
)

df_model.sort_values(
    ["StationId", "Datetime"],
    inplace=True
)

df_model.reset_index(drop=True, inplace=True)

print("Loaded shape:", df_model.shape)
print("Date range:", df_model["Datetime"].min(), "to",
      df_model["Datetime"].max())
print("Available measurements:",
      [c for c in candidate_features if c in df_model.columns])

Loaded shape: (1865486, 14)
Date range: 2015-01-01 16:00:00 to 2020-06-29 00:00:00
Available measurements: ['PM2.5', 'PM10', 'NO2', 'SO2', 'O3', 'CO', 'Temperature_C', 'Relative_Humidity_pct', 'Wind_Speed_kmh']


## 3. Create simple time features and choose model inputs

For this first baseline, we use current AQI, available pollutant readings, available weather measurements, and calendar features. We exclude station IDs and text fields from the numeric model inputs. More advanced lag and rolling features can be added after this baseline works.

In [7]:
# --------------------------------------------------
# 2. Create time features
# --------------------------------------------------

df_model["Hour"] = df_model["Datetime"].dt.hour
df_model["DayOfWeek"] = df_model["Datetime"].dt.dayofweek
df_model["Month"] = df_model["Datetime"].dt.month
df_model["DayOfYear"] = df_model["Datetime"].dt.dayofyear

# Cyclical encodings preserve the circular nature of time.
df_model["Hour_sin"] = np.sin(2 * np.pi * df_model["Hour"] / 24)
df_model["Hour_cos"] = np.cos(2 * np.pi * df_model["Hour"] / 24)

df_model["DayOfWeek_sin"] = np.sin(
    2 * np.pi * df_model["DayOfWeek"] / 7
)
df_model["DayOfWeek_cos"] = np.cos(
    2 * np.pi * df_model["DayOfWeek"] / 7
)

df_model["Month_sin"] = np.sin(
    2 * np.pi * (df_model["Month"] - 1) / 12
)
df_model["Month_cos"] = np.cos(
    2 * np.pi * (df_model["Month"] - 1) / 12
)


In [8]:
# --------------------------------------------------
# 3. Create hourly AQI lag features
# --------------------------------------------------

station_aqi = df_model.groupby("StationId", sort=False)["AQI_final"]

for lag in [1, 6, 12, 24]:
    df_model[f"AQI_lag_{lag}h"] = station_aqi.shift(lag)


In [9]:
# --------------------------------------------------
# 4. Create rolling AQI averages
# --------------------------------------------------

# Rolling windows are calculated separately for each station.
# min_periods requires a complete window of observations.
for window in [6, 12, 24]:
    df_model[f"AQI_rolling_mean_{window}h"] = (
        df_model.groupby("StationId", sort=False)["AQI_final"]
        .transform(
            lambda s: s.rolling(
                window=window,
                min_periods=window
            ).mean()
        )
    )

In [10]:
# --------------------------------------------------
# 5. Select model features and forecasting targets
# --------------------------------------------------

time_features = [
    "Hour_sin", "Hour_cos",
    "DayOfWeek_sin", "DayOfWeek_cos",
    "Month_sin", "Month_cos",
    "DayOfYear"
]

lag_features = [
    f"AQI_lag_{lag}h" for lag in [1, 6, 12, 24]
]

rolling_features = [
    f"AQI_rolling_mean_{window}h" for window in [6, 12, 24]
]

measurement_features = [
    col for col in candidate_features if col in df_model.columns
]

feature_columns = (
    ["AQI_final"]
    + measurement_features
    + time_features
    + lag_features
    + rolling_features
)

target_columns = ["Target_AQI_24h", "Target_AQI_48h"]

print("Number of features:", len(feature_columns))
print("Features:", feature_columns)

# Rows without sufficient past AQI history cannot use all features.
# Drop these rows from the modelling dataset.
df_model.dropna(
    subset=feature_columns + target_columns,
    inplace=True
)

df_model.reset_index(drop=True, inplace=True)

print("Shape after feature engineering:", df_model.shape)

Number of features: 24
Features: ['AQI_final', 'PM2.5', 'PM10', 'NO2', 'SO2', 'O3', 'CO', 'Temperature_C', 'Relative_Humidity_pct', 'Wind_Speed_kmh', 'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'DayOfYear', 'AQI_lag_1h', 'AQI_lag_6h', 'AQI_lag_12h', 'AQI_lag_24h', 'AQI_rolling_mean_6h', 'AQI_rolling_mean_12h', 'AQI_rolling_mean_24h']
Shape after feature engineering: (979793, 31)


In [16]:
df_model.head()

,StationId,Datetime,PM2.5,PM10,NO2,CO,SO2,O3,Temperature_C,Relative_Humidity_pct,...,DayOfWeek_cos,Month_sin,Month_cos,AQI_lag_1h,AQI_lag_6h,AQI_lag_12h,AQI_lag_24h,AQI_rolling_mean_6h,AQI_rolling_mean_12h,AQI_rolling_mean_24h
0,AP001,2017-11-29 14:00:00,31.75,63.25,12.70,0.10,11.88,137.70,32.4,32,...,-0.222521,-0.866025,0.5,157.0,160.0,186.0,173.0,158.833333,157.833333,173.625000
1,AP001,2017-11-29 15:00:00,37.00,67.25,14.35,0.10,11.85,153.85,32.0,34,...,-0.222521,-0.866025,0.5,155.0,163.0,186.0,184.0,161.500000,157.250000,173.416667
2,AP001,2017-11-29 16:00:00,48.75,81.50,20.15,0.10,10.75,164.62,29.5,51,...,-0.222521,-0.866025,0.5,179.0,163.0,140.0,184.0,166.833333,161.833333,173.875000
3,AP001,2017-11-29 17:00:00,59.25,100.50,26.20,0.25,11.55,165.12,27.7,56,...,-0.222521,-0.866025,0.5,195.0,160.0,146.0,191.0,172.833333,166.000000,174.083333
4,AP001,2017-11-29 18:00:00,69.50,125.50,36.67,0.25,12.75,141.12,26.3,61,...,-0.222521,-0.866025,0.5,196.0,155.0,152.0,191.0,179.666667,169.666667,174.291667


## 4. Split chronologically

We split by timestamp rather than randomly, so later observations are held out for testing. The small 48-hour gap before the test period helps prevent training targets from reaching into the test period.

In [11]:
# --------------------------------------------------
# 6. Chronological train-test split
# --------------------------------------------------

unique_times = np.sort(df_model["Datetime"].unique())
split_position = int(len(unique_times) * 0.8)
split_time = pd.Timestamp(unique_times[split_position])

# Keep a 48-hour gap to reduce overlap from forecast targets.
train_end_time = split_time - pd.Timedelta(hours=48)

train_mask = df_model["Datetime"] < train_end_time
test_mask = df_model["Datetime"] >= split_time

X_train = df_model.loc[train_mask, feature_columns]
X_test = df_model.loc[test_mask, feature_columns]

y_train_24h = df_model.loc[train_mask, "Target_AQI_24h"]
y_test_24h = df_model.loc[test_mask, "Target_AQI_24h"]

y_train_48h = df_model.loc[train_mask, "Target_AQI_48h"]
y_test_48h = df_model.loc[test_mask, "Target_AQI_48h"]

print("Split timestamp:", split_time)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Split timestamp: 2019-06-20 19:00:00
Training rows: 505018
Testing rows: 472425


## 5. Impute missing feature values without data leakage

The median values are learned from the training data only. We then apply those same values to the test data. This avoids using test-set information during training.

In [12]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)


print("Training matrix shape:", X_train_imputed.shape)
print("Testing matrix shape:", X_test_imputed.shape)
print("Remaining missing values in training:", np.isnan(X_train_imputed).sum())
print("Remaining missing values in testing:", np.isnan(X_test_imputed).sum())

Training matrix shape: (505018, 24)
Testing matrix shape: (472425, 24)
Remaining missing values in training: 0
Remaining missing values in testing: 0


# Train and evaluate Linear Regression — 24-hour forecast

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_24h = LinearRegression()
model_24h.fit(X_train_imputed, y_train_24h)

pred_24h = model_24h.predict(X_test_imputed)

mae_24h = mean_absolute_error(y_test_24h, pred_24h)
rmse_24h = np.sqrt(mean_squared_error(y_test_24h, pred_24h))
r2_24h = r2_score(y_test_24h, pred_24h)

print("24-hour Linear Regression results")
print(f"MAE:  {mae_24h:.3f}")
print(f"RMSE: {rmse_24h:.3f}")
print(f"R²:   {r2_24h:.3f}")

24-hour Linear Regression results
MAE:  32.440
RMSE: 52.766
R²:   0.799


## 7. Train and evaluate Linear Regression — 48-hour forecast

In [14]:
model_48h = LinearRegression()
model_48h.fit(X_train_imputed, y_train_48h)

pred_48h = model_48h.predict(X_test_imputed)

mae_48h = mean_absolute_error(y_test_48h, pred_48h)
rmse_48h = np.sqrt(mean_squared_error(y_test_48h, pred_48h))
r2_48h = r2_score(y_test_48h, pred_48h)

print("48-hour Linear Regression results")
print(f"MAE:  {mae_48h:.3f}")
print(f"RMSE: {rmse_48h:.3f}")
print(f"R²:   {r2_48h:.3f}")

48-hour Linear Regression results
MAE:  43.093
RMSE: 66.956
R²:   0.682


## 8. Compare the two baselines

MAE and RMSE are prediction-error measures; lower is better. R² compares the model with a simple mean-prediction baseline; closer to 1 is generally better.

In [15]:
results = pd.DataFrame({
    "Forecast horizon": ["24 hours", "48 hours"],
    "MAE": [mae_24h, mae_48h],
    "RMSE": [rmse_24h, rmse_48h],
    "R2": [r2_24h, r2_48h]
})

results

,Forecast horizon,MAE,RMSE,R2
0,24 hours,32.440435,52.765805,0.798546
1,48 hours,43.093182,66.956118,0.681783


## Important notes

- This is the first Linear Regression baseline, not the final forecasting system.
- `shift(-24)` and `shift(-48)` are row-based shifts. Missing hourly records can mean these are not exactly 24 or 48 clock-hours ahead.
- The current baseline uses current pollutant/weather readings and calendar features. Project-specified lag and rolling-average features are a sensible next improvement after this notebook runs successfully.
- The original `04_DSA.ipynb` and its DSA plots/code remain untouched in your project. This working copy avoids the old test-set re-imputation and the repeated plotting cells that referenced `Target_AQI_Tomorrow`, which no longer matches the two-target setup.